# Codestral Mamba 7B (FP4-KM + LoRA) trên Colab T4 bằng Vulkan

Chọn **Runtime → Change runtime type → T4 GPU**, rồi chạy lần lượt từ trên xuống. Notebook cài Nim 2.2.10 và Vulkan, clone repo kèm submodule, tải `codestral_mamba_only.nimq`, tokenizer và adapter LoRA từ Hugging Face, build `infer_lora` và `formers-cli`, rồi chạy inference trên Vulkan. Tokenizer chạy thuần Nim.

LoRA luôn được merge vào weight FP4 ngay trên GPU, nên decode không còn phép cộng adapter. Prefill dùng Tensor Core của T4 (`KHR_cooperative_matrix`, FP16 nhân, FP32 cộng dồn). FP4 chỉ là định dạng lưu trữ. Decode là GEMV FP4, được autotune theo từng shape.

Sửa `PROMPT`, `MAX_NEW_TOKENS` ở cell inference.

In [1]:
# 1) Kiểm tra runtime GPU và cài các gói hệ thống, gồm Vulkan SDK/runtime và shader compiler.
import os, subprocess, sys
from pathlib import Path

assert Path('/content').exists(), 'Notebook này cần chạy trên Google Colab.'
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'], text=True))
except Exception as e:
    raise RuntimeError('Chưa có GPU NVIDIA. Chọn Runtime → Change runtime type → T4 GPU rồi chạy lại.') from e

subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','build-essential','git','curl','ca-certificates','xz-utils','pkg-config','time','libvulkan1','libvulkan-dev','vulkan-tools','glslc','libegl1','libglx0','libx11-6','libxext6','libx11-xcb1','libxcb1','libxcb-dri3-0','libxcb-present0','libxfixes3','libxrandr2','libdrm2','libexpat1','libwayland-client0'], check=True)


Tesla T4, 580.82.07, 15360 MiB



CompletedProcess(args=['apt-get', 'install', '-y', '-qq', 'build-essential', 'git', 'curl', 'ca-certificates', 'xz-utils', 'pkg-config', 'time', 'libvulkan1', 'libvulkan-dev', 'vulkan-tools', 'glslc', 'libegl1', 'libglx0', 'libx11-6', 'libxext6', 'libx11-xcb1', 'libxcb1', 'libxcb-dri3-0', 'libxcb-present0', 'libxfixes3', 'libxrandr2', 'libdrm2', 'libexpat1', 'libwayland-client0'], returncode=0)

In [2]:
# 2) Cài Nim 2.2.10 chính chủ và xác minh checksum (tokenizer thuần Nim).
import hashlib, urllib.request, tarfile, os, subprocess
from pathlib import Path

NIM_VERSION = '2.2.10'
TOOLS = Path('/content/tools'); TOOLS.mkdir(exist_ok=True)
archive = TOOLS / f'nim-{NIM_VERSION}-linux_x64.tar.xz'
url = f'https://nim-lang.org/download/nim-{NIM_VERSION}-linux_x64.tar.xz'

# Tải file với User-Agent giả lập trình duyệt để tránh lỗi 403
if not archive.exists():
    req = urllib.request.Request(
        url,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
    )
    with urllib.request.urlopen(req) as response, open(archive, 'wb') as out_file:
        out_file.write(response.read())

# Xác minh checksum
expected = '0a3a38752e97e9d44aa479b3a7b37336dfe0176daf22ee5b5218ad0991ecd211'
actual = hashlib.sha256(archive.read_bytes()).hexdigest()
assert actual == expected, f'Nim checksum mismatch: {actual}'

# Giải nén
with tarfile.open(archive, 'r:xz') as tf:
    tf.extractall(TOOLS, filter='data')

# Cấu hình PATH
NIM_HOME = TOOLS / f'nim-{NIM_VERSION}'
os.environ['PATH'] = f"{NIM_HOME}/bin:" + os.environ['PATH']

# Kiểm tra phiên bản
subprocess.run(['nim', '--version'], check=True)


CompletedProcess(args=['nim', '--version'], returncode=0)

In [3]:
# 3) Clone repository bản main mới nhất và tải submodule đã ghim trong repo.
import os, subprocess
from pathlib import Path

URL = 'https://github.com/bobbyshop-vui/nimformers-framework.git'
BRANCH = 'main'
REPO = Path('/content/nimformers-framework')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
subprocess.run(['git', '-C', str(REPO), 'submodule', 'sync', '--recursive'], check=True)
subprocess.run(['git', '-C', str(REPO), 'submodule', 'update', '--init', '--recursive'], check=True)
os.chdir(REPO)
print('Repo:', REPO, '| branch:', BRANCH)
print(subprocess.check_output(['git', 'log', '--oneline', '-1'], text=True))

Repo: /content/nimformers-framework | branch: main
5c1e6e3 Keep Codestral setup out of FIR replay



In [4]:
# 4) Cài Vulkan + kiểm tra ICD NVIDIA và xác nhận dGPU (không chấp nhận lavapipe/CPU)
import subprocess, os, re
from pathlib import Path

# --- Bước 1: Cài vulkan-tools ---
print("[1/4] Cài vulkan-tools...")
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'vulkan-tools'], check=True)

# --- Bước 2: Xác định phiên bản driver NVIDIA và cài libnvidia-gl tương ứng ---
print("[2/4] Phát hiện phiên bản driver NVIDIA...")
smi = subprocess.check_output(['nvidia-smi'], text=True)
m = re.search(r'Driver Version:\s*(\d+)\.', smi)
assert m, "Không đọc được Driver Version từ nvidia-smi"
driver_major = m.group(1)
print(f"    -> Driver major: {driver_major}")

pkg = f'libnvidia-gl-{driver_major}'
print(f"[2/4] Cài {pkg}...")
subprocess.run(['apt-get', 'install', '-y', '-qq', pkg], check=True)

# --- Bước 3: Đảm bảo file ICD NVIDIA có thể được loader tìm thấy ---
print("[3/4] Cấu hình Vulkan ICD...")
Path('/usr/share/vulkan/icd.d').mkdir(parents=True, exist_ok=True)
candidates = [
    '/etc/vulkan/icd.d/nvidia_icd.json',
    '/usr/share/vulkan/icd.d/nvidia_icd.json',
    f'/usr/lib/x86_64-linux-gnu/libnvidia-gl-{driver_major}/nvidia_icd.json',
]
src = next((c for c in candidates if Path(c).exists()), None)
if src and src != '/usr/share/vulkan/icd.d/nvidia_icd.json':
    subprocess.run(['ln', '-sf', src, '/usr/share/vulkan/icd.d/nvidia_icd.json'], check=True)
    print(f"    -> symlink: {src} -> /usr/share/vulkan/icd.d/nvidia_icd.json")
else:
    print(f"    -> ICD có sẵn: {src}")

# --- Bước 4: Chạy kiểm tra ---
print("[4/4] Kiểm tra Vulkan...")
subprocess.run(['vulkaninfo', '--summary'], check=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
summary = subprocess.check_output(['vulkaninfo', '--summary'],
                                   text=True, stderr=subprocess.STDOUT)
print(''.join(line for line in summary.splitlines()
              if any(k in line.lower() for k in ('device name', 'device type', 'driver name'))))

assert 'NVIDIA' in summary.upper(), \
    'Không thấy NVIDIA Vulkan ICD. Runtime GPU hiện tại không expose Vulkan graphics.'
assert 'DISCRETE_GPU' in summary.upper(), \
    'Không tìm thấy NVIDIA dGPU qua Vulkan; dừng để tránh fallback CPU/iGPU.'

print("✅ Vulkan OK: NVIDIA dGPU đã được expose.")

[1/4] Cài vulkan-tools...
[2/4] Phát hiện phiên bản driver NVIDIA...
    -> Driver major: 580
[2/4] Cài libnvidia-gl-580...
[3/4] Cấu hình Vulkan ICD...
    -> ICD có sẵn: /usr/share/vulkan/icd.d/nvidia_icd.json
[4/4] Kiểm tra Vulkan...

✅ Vulkan OK: NVIDIA dGPU đã được expose.


In [5]:
# 5) Download the Codestral FP4-KM base, tokenizer and LoRA adapter. Hugging Face resumes interrupted downloads.
import subprocess, sys
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'huggingface_hub'], check=True)
from huggingface_hub import hf_hub_download, model_info

MODELS = REPO / 'test/codestral/models'
HF_DIR = MODELS / 'hf'; HF_DIR.mkdir(parents=True, exist_ok=True)
BASE_REPO = 'bobbydeveloper2014/codestral-v0.1-7b-nimq'
BASE_REV = '148dd26885b5eeef03476683f05f0a4546cd75ba'
for name in ['codestral_mamba_only.nimq', 'config.json', 'generation_config.json', 'params.json',
             'special_tokens_map.json', 'tokenizer.json', 'tokenizer.model', 'tokenizer.model.v3', 'tokenizer_config.json']:
    dest = HF_DIR if name.endswith('.json') or name.startswith('tokenizer.') else MODELS
    hf_hub_download(BASE_REPO, filename=name, revision=BASE_REV, local_dir=str(dest))
ADAPTER_REPO = 'bobbydeveloper2014/codestral-v0.1-nimq-qlora-linearattention'
ADAPTER_REV = model_info(ADAPTER_REPO).sha
adapter = hf_hub_download(ADAPTER_REPO, filename='jamba_parallel_lora.bin', revision=ADAPTER_REV, local_dir=str(MODELS))
Path(str(adapter) + '.hf_tokenizer_dir').write_text(str(HF_DIR.resolve()))
print('Adapter revision:', ADAPTER_REV)
for f in sorted(MODELS.iterdir()):
    if f.is_file():
        print(f'{f.name:40} {f.stat().st_size / 2**20:10.1f} MB')

codestral_mamba_only.nimq: reconstructing file:   0%|          |  0.00B / 4.34GB            

codestral_mamba_only.nimq: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

params.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer_config.json:   0%|          | 0.00/137k [00:00<?, ?B/s]

jamba_parallel_lora.bin: reconstructing file:   0%|          |  0.00B / 1.65GB            

jamba_parallel_lora.bin: downloading bytes:           |  0.00B            

Adapter revision: 6039f2dc2e6a3c7df3a1965eca5f9df112a9101e
codestral_mamba_only.nimq                    4141.1 MB
jamba_parallel_lora.bin                      1570.5 MB
jamba_parallel_lora.bin.hf_tokenizer_dir        0.0 MB


In [6]:
# 6) Compile the Vulkan shaders and the infer_lora binary.
import os
os.chdir(str(REPO))

!make shaders
!make build-infer-lora

[shaders] done.
nim check -d:release --hints:off --mm:arc --opt:speed --path:. --path:test/codestral --path:test/deepseek-inference --path:test/spectron --path:kernelfortest kernelfortest/fp4_gemv_gpu.nim
nim check -d:release --hints:off --mm:arc --opt:speed --path:. --path:test/codestral --path:test/deepseek-inference --path:test/spectron --path:kernelfortest -d:codestralInference test/codestral/infer_lora.nim
/content/nimformers-framework/test/codestral/jamba_hybrid.nim(22, 8) Warning: imported and not used: 'customfloat' [UnusedImport]
/content/nimformers-framework/test/codestral/jamba_hybrid.nim(21, 19) Warning: imported and not used: 'random' [UnusedImport]
/content/nimformers-framework/test/codestral/onedispatch.nim(167, 18) Warning: loraAIn, loraBIn, loraAOut, loraBOut all have default value '@[]', this may be unintentional, either use ';' (semicolon) or explicitly write each default value [ImplicitDefaultValue]
/content/nimformers-framework/test/codestral/onedispatch.nim(24, 8)

## Inference

`codestral_mamba_only.nimq` được mmap thẳng từ đĩa. Trên T4, mỗi weight được copy một lần vào VRAM (`BYBY_VULKAN_DEVICE_LOCAL=auto`). Sau đó adapter LoRA được merge vào weight FP4 trên GPU. Log in thời gian merge của từng tensor.

Dòng `[vulkan] ... matrix=KHR_cooperative_matrix FP16/FP32 prefill=tensor-cores-only` xác nhận prefill chạy trên Tensor Core. Lần chạy đầu phải biên dịch shader bằng glslc nên chậm hơn; các lần sau dùng cache.

In [7]:
# 7) Run inference (merged LoRA, greedy).
import os

PROMPT = 'You are a senior software engineer. Write a complete Python program to manage a library of books that meets these requirements: (1) Define a Book class with attributes: id, title, author, year, available (bool). (2) Define a Library class holding a list of books, with methods: add_book, remove_book(id), find_by_author(author), borrow_book(id), return_book(id), and list_available(). (3) Handle exceptions when borrowing an already-borrowed or nonexistent book, printing clear error messages. (4) Write a main() function demonstrating adding 5 books, borrowing/returning, searching by author, and listing available books. (5) Ensure full type hints, docstrings for every class and method, and PEP8 compliance. (6) Add unit tests using unittest for at least 4 methods, including error cases. (7) Briefly explain the time complexity of each method and suggest improvements if the list holds millions of books. Present the code in separate markdown blocks for the main code and the tests, with explanations in English.'
MAX_NEW_TOKENS = 999
model = MODELS / 'jamba_parallel_lora.bin'
base = MODELS / 'codestral_mamba_only.nimq'
assert model.exists() and base.exists() and (HF_DIR / 'tokenizer.model.v3').exists()
INFER_PROMPT = f'[INST] {PROMPT}[/INST]'

os.chdir(str(REPO))
print('Backend Vulkan | mmap | merged LoRA | max tokens =', MAX_NEW_TOKENS)

!BYBY_INFER_BACKEND=vulkan \
 BYBY_PROGRESS=0 \
 BYBY_MODEL_MMAP=1 BYBY_VULKAN_DEVICE_LOCAL=auto BYBY_VULKAN_TENSOR_CORES=1 \
 BYBY_MAX_NEW_TOKENS={MAX_NEW_TOKENS} \
 ./infer_lora_bin "{model}" "{INFER_PROMPT}" 0.0 "{HF_DIR}" "{base}"

Backend Vulkan | mmap | merged LoRA | max tokens = 999
[infer] GEMV_TG=auto autotune=1 debug_sampling=0; prefill GEMM=auto; prefill scan=auto; FP4 2col=1; FP4 kernel=auto; resident prefill=1; decode=native (no MPS)
== [Framework] Backend selected: vulkan ==
[infer] Detected NIMBIN4 (QLoRA 4-bit) — newBlock stays compressed, dequantized on the fly during generate()...
[infer] checkpoint records the training norm preset: rms-fin-gn
[infer] checkpoint records the Mamba norm implementation: mamba-ssm-postgate-grouped-v1
[infer] Pattern block (64 block): RRRRRRRGRRRRRRRRRRRRRRRGRRRRRRRRRRRRRRRGRRRRRRRRRRRRRRRGRRRRRRRR
[infer] Thanh phan: R(Mamba2Real)=60 P(Mamba2Real+Linear++ sum)=0 G(Mamba2Real+gated Linear++ fuseGate)=4 M(Mamba)=0 A(LinearPlusAttention)=0 S(Spectron)=0
[hf_tokenizer] pure Nim BPE, vocab=32768, normalizer=identity, bos=1 eos=2 unk=0 stopIds=@[2, 3, 4]
[load_codestral] (quant-resident) Doc /content/nimformers-framework/test/codestral/models/codestral_mamba_only.nimq ...
[lo

## formers.n

Chạy cùng model bằng `formers-cli`. Nó tự nhận diện Mamba-2 và LoRA từ `hf/config.json`, header `.nimq` và `jamba_parallel_lora.bin`, merge LoRA trên GPU, rồi dùng nhánh checkpoint của `infer_lora`. Prompt thường được bọc thành `[INST] ...[/INST]`. Với `--verbose`, log có thêm các dòng `[vulkan] ... prefill=tensor-cores-only` và `[vulkan] mamba2 SSD prefill ...`. Phần text sinh ra là mốc để so với graph FIR ở cuối notebook.

In [8]:
# Rebuild formers-cli and measure Codestral on the T4 (FP16 sidecar tile GEMM + coop prefetch).
import os
os.chdir(str(REPO))
!make formers-cli
!BYBY_MODEL_MMAP=1 BYBY_VULKAN_DEVICE_LOCAL=auto BYBY_VULKAN_TENSOR_CORES=1 \
 other/formers.n/formers-cli --models "{MODELS}" --backend vulkan -p "{PROMPT}" -n 200 --temp 0 --repeat-penalty 1.0 --no-progress --verbose


make -C other/formers.n build
make[1]: Entering directory '/content/nimformers-framework/other/formers.n'
make -C /content/nimformers-framework/vendor/bybylang build
make[2]: Entering directory '/content/nimformers-framework/vendor/bybylang'
[Makefile] OS: Linux
nim c -d:release -o:bybylang bybylang.nim
Hint: used config file '/content/tools/nim-2.2.10/config/nim.cfg' [Conf]
Hint: used config file '/content/tools/nim-2.2.10/config/config.nims' [Conf]
...................................................................................................................................................
/content/nimformers-framework/vendor/bybylang/backends/cpu/cpu_backend.nim(56, 6) Hint: 'nimf_cpu_current_isa' is declared but not used [XDeclaredButNotUsed]
.
/content/nimformers-framework/vendor/bybylang/backends/metal/metal_backend.nim(6, 6) Hint: 'c_free' is declared but not used [XDeclaredButNotUsed]
.....
/content/nimformers-framework/vendor/bybylang/backends/vulkan/vulkan_driver.nim(3662

### Trace và chạy lifted FIR trên Tesla T4

Cell đầu chạy runtime Vulkan dưới `firlift --run` với LoRA đã merge, trên một prompt đúng 254 token. Như vậy mỗi kích thước prefill (128, 64, 32, 16, 8, 5 hàng) thành một phase, và bước decode cuối được lift thành `phase decode`. Trace đi qua pipe, không ghi file `.fprobe`. Thư mục output chỉ có `model.fir`, `config.json`, `trace.data` (khoảng 4 GB weight đã merge) và symlink tới adapter. Trên trace có `coopmatrix`, GEMM FP4 của prefill được lift thành micro-kernel Tensor Core `fp4_gemm_tensor_core_micro`.

Cell kế tiếp chạy graph đó trên cùng `PROMPT`, greedy, backend Vulkan. Token id được tạo bằng tokenizer native, nên khớp với cell formers.n ở trên. Tốc độ prefill và decode của graph được in riêng.

In [9]:
import os
import shlex
from pathlib import Path

repo = Path(REPO).resolve()
os.chdir(repo)

FIR_CONTEXT = 512
FIR_TOKENS = 200
FIR_OUT = Path("/content/codestral-t4-fir-resident")
FIR_MODEL = FIR_OUT / "model.fir"
FIR_CONFIG = FIR_OUT / "config.json"
HF_CONFIG = Path(HF_DIR) / "config.json"

FIRLIFT = repo / "other/formers.n/firlift/firlift"
FORMERS_CLI = repo / "other/formers.n/formers-cli"

for binary in (FIRLIFT, FORMERS_CLI):
    if not binary.is_file() or not os.access(binary, os.X_OK):
        raise RuntimeError(
            f"{binary} is missing or not executable. Run !make formers-cli in the build cell."
        )

common_args = [
    str(FORMERS_CLI),
    "--models", str(MODELS),
    "-m", str(base),
    "--lora", str(model),
    "--tokenizer", str(HF_DIR),
    "--backend", "vulkan",
    "-p", PROMPT,
    "--temp", "0",
    "--repeat-penalty", "1.0",
    "--no-progress",
]

lift_cmd = shlex.join([
    str(FIRLIFT),
    "--config", str(HF_CONFIG),
    "--weights", str(base),
    "--context", str(FIR_CONTEXT),
    "--out", str(FIR_OUT),
    "--report",
    "--run",
    *common_args,
    "--config", str(HF_CONFIG),
    "-n", "4",
])

!{lift_cmd}
if _exit_code:
    raise RuntimeError(f"FIR lift failed with exit code {_exit_code}")

assert {p.name for p in FIR_OUT.iterdir()} == {"model.fir", "config.json"}

run_cmd = shlex.join([
    *common_args,
    "--config", str(FIR_CONFIG),
    "--fir", str(FIR_MODEL),
    "-c", str(FIR_CONTEXT),
    "-n", str(FIR_TOKENS),
])

!{run_cmd}
if _exit_code:
    raise RuntimeError(f"FIR run failed with exit code {_exit_code}")

[formers.n] mamba2+linear++ /content/nimformers-framework/test/codestral/models/codestral_mamba_only.nimq + jamba_parallel_lora.bin - loading (logs: --verbose)
[INST] You are a senior software engineer. Write a complete Python program to manage a library of books that meets these requirements: (1) Define a Book class with attributes: id, title, author, year, available (bool). (2) Define a Library class holding a list of books, with methods: add_book, remove_book(id), find_by_author(author), borrow_book(id), return_book(id), and list_available(). (3) Handle exceptions when borrowing an already-borrowed or nonexistent book, printing clear error messages. (4) Write a main() function demonstrating adding 5 books, borrowing/returning, searching by author, and listing available books. (5) Ensure full type hints, docstrings for every class and method, and PEP8 compliance. (6) Add unit tests using unittest for at least 4 methods, including error cases. (7) Briefly explain the time complexity o

In [11]:
import os
import shlex
from pathlib import Path

repo = Path(REPO).resolve()
fir_dir = Path(FIR_OUT)
os.chdir(repo)
FIR_TOKENS = 200

env_args = [
    "env",
    "-u", "BYBY_FIR_DECODE_REPLAY",
    "-u", "BYBY_FIR_DECODE_CB_NODES",
    "-u", "BYBY_VULKAN_COMMIT_WAIT",
    "-u", "BYBY_FIR_FP4_GEMV_TUNE",
    "-u", "BYBY_PREFILL_MIN_ROWS",
    "BYBY_MEMORY_LIMIT_MB=10000",
    "BYBY_VULKAN_DEVICE_LOCAL=auto",
    "BYBY_VULKAN_TENSOR_CORES=1",
    "BYBY_UBATCH=512",
    "BYBY_SSD=auto",
    "BYBY_CODESTRAL_TOKEN_IDS=1",
    "BYBY_FIR_PROFILE=1",
]

cli_args = [
    str(repo / "other/formers.n/formers-cli"),
    "--models", str(MODELS),
    "-m", str(base),
    "--lora", str(model),
    "--config", str(fir_dir / "config.json"),
    "--tokenizer", str(HF_DIR),
    "--fir", str(fir_dir / "model.fir"),
    "--backend", "vulkan",
    "-p", PROMPT,
    "-n", str(FIR_TOKENS),
    "--temp", "0",
    "--repeat-penalty", "1.0",
    "--no-progress",
    "-c", str(FIR_CONTEXT),
]

run_cmd = shlex.join(env_args + cli_args)
!{run_cmd}
if _exit_code:
    raise RuntimeError(f"FIR run failed with exit code {_exit_code}")

[formers.n] fir /content/nimformers-framework/test/codestral/models/codestral_mamba_only.nimq + jamba_parallel_lora.bin - loading (logs: --verbose)
[formers.n] FIR graph recorded prefill rows: 246
[formers.n] ready: warmup 1.2 s (kernel setup/JIT, prompt + 2 decode steps)


Here is the complete Python program to manage a library of books:

```python
class Book:
    """Represents a book with attributes: id, title, author, year, and availability."""

    def __init__(self, id: int, title: str, author: str, year: int, available: bool):
        self.id = id
        self.title = title
        self.author = author
        self.year = year
        self.available = available

class Library:
    """Represents a library holding a list of books and provides methods to manage them."""

    def __init__(self):
        self.books = []

    def add_book(self, book: Book):
        """Adds a book to the library."""
        self.books.append(book)

    def remove_book(self, id: int):
       
[codestral-

### Tài liệu

- [Nim 2.2.10](https://nim-lang.org/download/)
- [Google Colab FAQ: GPU runtime](https://research.google.com/colaboratory/faq.html)
- [VK_KHR_cooperative_matrix](https://registry.khronos.org/vulkan/specs/latest/man/html/VK_KHR_cooperative_matrix.html)
- [Codestral NimQ trên Hugging Face](https://huggingface.co/bobbydeveloper2014/codestral-v0.1-7b-nimq)